# FVF-based SDT measures

`utils/sdt.py` divides false alarms by `num_distractors` - *every* non-target icon (~177 per trial), which
treats each one as an independent response opportunity, including the majority the subject never looked at
(`CODE_REVIEW.md` T1). This notebook prototypes the proposed fix: restrict both the hit-rate and FA-rate
denominators to items that actually fell within the subject's FVF along the scanpath (from
`array_coverage.ipynb`), and compares the result against the existing unconditioned rates.

**Prototype only - `utils/sdt.py` is not modified here.**

In [ ]:
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio

import constants as cnst
import config as cnfg
from analysis.helpers.read_data import load_data
from pipeline.stage2_align.fixations_to_icons import fixations_to_icons
from analysis.fvf.fvf import estimate_fvf
from data_models.LWSEnums import SignalDetectionCategoryEnum
from utils.sdt import calc_sdt_metrics

pio.renderers.default = 'notebook'      # 'notebook' or 'browser'

### Per-trial targets/distractors within FVF

Same computation as `array_coverage.ipynb`, but split by `is_target` instead of collapsed into one coverage
number - the FVF-based denominators need targets and distractors counted separately.

In [ ]:
loaded_data = load_data(cnfg.OUTPUT_PATH)
fixations = loaded_data.fixations
icons = loaded_data.icons
metadata = loaded_data.metadata
idents = loaded_data.identifications

fvf_by_subject = estimate_fvf(loaded_data.fixation_target_dists)['selection_hazard'].drop(index='all')

all_icon_dists = fixations_to_icons(fixations, icons, metadata)
closest = (
    all_icon_dists
    .groupby([cnst.SUBJECT_STR, cnst.TRIAL_STR, cnst.ICON_STR], observed=True)[cnst.DISTANCE_DVA_STR]
    .min()
    .reset_index()
)
closest['fvf_radius'] = closest[cnst.SUBJECT_STR].map(fvf_by_subject)
closest = closest.dropna(subset=['fvf_radius'])
closest['within_fvf'] = closest[cnst.DISTANCE_DVA_STR] <= closest['fvf_radius']

is_target_lookup = icons.set_index([cnst.SUBJECT_STR, cnst.TRIAL_STR, cnst.ICON_STR])['is_target']
closest['is_target'] = closest.set_index(
    [cnst.SUBJECT_STR, cnst.TRIAL_STR, cnst.ICON_STR]
).index.map(is_target_lookup)

fvf_counts = (
    closest
    .groupby([cnst.SUBJECT_STR, cnst.TRIAL_STR, 'is_target'], observed=True)['within_fvf']
    .sum()
    .unstack('is_target', fill_value=0)
    .rename(columns={True: 'targets_in_fvf', False: 'distractors_in_fvf'})
    .reset_index()
)
fvf_counts.head()

### FVF-conditioned hit/FA rates

In [ ]:
hit_counts = (
    idents.loc[idents[cnst.IDENTIFICATION_CATEGORY_STR] == SignalDetectionCategoryEnum.HIT]
    .groupby([cnst.SUBJECT_STR, cnst.TRIAL_STR], observed=True)
    .size()
    .rename('hits')
)
fa_counts = (
    idents.loc[idents[cnst.IDENTIFICATION_CATEGORY_STR] == SignalDetectionCategoryEnum.FALSE_ALARM]
    .groupby([cnst.SUBJECT_STR, cnst.TRIAL_STR], observed=True)
    .size()
    .rename('false_alarms')
)

fvf_sdt = fvf_counts.merge(hit_counts, on=[cnst.SUBJECT_STR, cnst.TRIAL_STR], how='left')
fvf_sdt = fvf_sdt.merge(fa_counts, on=[cnst.SUBJECT_STR, cnst.TRIAL_STR], how='left')
fvf_sdt[['hits', 'false_alarms']] = fvf_sdt[['hits', 'false_alarms']].fillna(0)

fvf_sdt['hit_rate_fvf'] = fvf_sdt['hits'] / fvf_sdt['targets_in_fvf'].replace(0, np.nan)
fvf_sdt['false_alarm_rate_fvf'] = fvf_sdt['false_alarms'] / fvf_sdt['distractors_in_fvf'].replace(0, np.nan)
fvf_sdt.head()

### Compare against the current (unconditioned) rates

In [ ]:
current_sdt = calc_sdt_metrics(metadata, idents, dprime_correction='loglinear')
comparison = fvf_sdt.merge(
    current_sdt[[cnst.SUBJECT_STR, cnst.TRIAL_STR, 'hit_rate', 'false_alarm_rate']],
    on=[cnst.SUBJECT_STR, cnst.TRIAL_STR], how='inner',
)
comparison[['hit_rate', 'hit_rate_fvf', 'false_alarm_rate', 'false_alarm_rate_fvf']].describe()

In [ ]:
fig = px.scatter(
    comparison, x='false_alarm_rate', y='false_alarm_rate_fvf', color=cnst.SUBJECT_STR,
    template='plotly_white', opacity=0.5,
    title='FA rate: all distractors vs. distractors within FVF',
    labels={'false_alarm_rate': 'FA rate (all distractors)', 'false_alarm_rate_fvf': 'FA rate (FVF-conditioned)'},
)
fig.add_shape(type='line', x0=0, y0=0, x1=comparison['false_alarm_rate_fvf'].max(),
             y1=comparison['false_alarm_rate_fvf'].max(), line=dict(dash='dot', color='grey'))
fig.show()